In [ ]:
import torch
import torch.nn as nn
import mlflow
import random
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.set_default_device(device)

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment(experiment_name="transformer_char_based")

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, seq_len):
        super().__init__()
        positions = torch.arange(0, seq_len) # (seq_len)
        inner = torch.arange(0, d_model, 2) # (d_model // 2)
        div_term = 10000 ** (inner / d_model) # (d_model // 2)
        divide = positions.unsqueeze(1) / div_term # (seq_len, d_model // 2)
        sin_ones = torch.sin(divide)
        cos_ones = torch.cos(divide)
        encoded_matrix = torch.stack((sin_ones, cos_ones), dim=2).reshape(seq_len, -1) # (seq_len, d_model)
        self.register_buffer("encoded_matrix", encoded_matrix)

    def forward(self, X):
        return X + self.encoded_matrix[:X.shape[-2]]

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, seq_len):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.softmax = nn.Softmax(dim=-1)
        self.num_heads = num_heads
        self.dim_model_single_head = d_model // self.num_heads
        initial_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer("initial_mask", initial_mask)

    def forward(self, X):
        Q = self.q_proj(X) # (batch_size, seq_len, d_model)
        V = self.v_proj(X)
        K = self.k_proj(X)

        Q_reshaped = Q.reshape(Q.shape[0], Q.shape[1], self.num_heads, -1).transpose(1, 2) # (batch_size, num_heads, seq_len, head_dim)
        V_reshaped = V.reshape(V.shape[0], V.shape[1], self.num_heads, -1).transpose(1, 2)
        K_reshaped = K.reshape(K.shape[0], K.shape[1], self.num_heads, -1).transpose(1, 2)

        initial = torch.matmul(Q_reshaped, K_reshaped.transpose(-1, -2)) # (batch_size, num_heads, seq_len, seq_len)

        initial = initial.masked_fill(self.initial_mask[:initial.shape[-1], :initial.shape[-1]], float("-inf"))

        normalized = self.softmax(torch.div(initial, torch.sqrt(torch.tensor(self.dim_model_single_head, dtype=torch.float32))))        

        with_values = torch.matmul(normalized, V_reshaped).transpose(1, 2).reshape(normalized.shape[0], normalized.shape[2], -1) # (batch_size, seq_len, d_model)

        return self.out_proj(with_values)


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj_hidden_layer = nn.Linear(d_model, d_model * 4)
        self.proj_outer_layer = nn.Linear(d_model * 4, d_model)
        self.relu = nn.ReLU()

    def forward(self, X):
        hidden_layer = self.relu(self.proj_hidden_layer(X))

        return self.proj_outer_layer(hidden_layer)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, X):
        matrix_mean = torch.mean(X, dim=-1, keepdims=True)
        matrix_standard_deviation = torch.std(X, dim=-1, keepdims=True)
        
        main_calculation = torch.div(torch.sub(X, matrix_mean), torch.add(matrix_standard_deviation, self.eps))

        scaled_calc = torch.mul(self.gamma, main_calculation)

        return torch.add(scaled_calc, self.beta)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, seq_len):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads, seq_len)
        self.first_layer_norm = LayerNorm(d_model)
        self.second_layer_norm = LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model)

    def forward(self, X):
        multi_head = self.multi_head_attention(X)

        residual_1 = multi_head + X
        layer_norm_1 = self.first_layer_norm(residual_1)

        feed_forward = self.feed_forward(layer_norm_1)

        residual_2 = feed_forward + layer_norm_1
        layer_norm_2 = self.second_layer_norm(residual_2)

        return layer_norm_2

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, seq_len):
        super().__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.proj_position_layer = PositionalEncoding(d_model=d_model, seq_len=seq_len)
        self.sequential = nn.Sequential(*[TransformerBlock(d_model=d_model, num_heads=num_heads, seq_len=seq_len) for _ in range(num_blocks)])
        self.proj_linear = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        embeded = self.embedding_layer(token_ids)

        embeded_with_position_encoding = self.proj_position_layer(embeded)

        transformer_blocks = self.sequential(embeded_with_position_encoding)

        return self.proj_linear(transformer_blocks)    

In [ ]:
# torch.manual_seed(42)
# model = Transformer(vocab_size=100, d_model=4, num_heads=2, num_blocks=2)

# token_ids = torch.tensor([1, 5, 17])
# output = model(token_ids)
# print(output.shape)
# print(output)

In [ ]:
def train_val_split(seed: int):
    train_split = "" 
    val_split = ""

    path = "./data"
    files = os.listdir(path)
    list_of_files = []

    for file in files:
        if file.endswith(".txt"):
            with open(f"{path}/{file}", "r", encoding="utf-8") as f:
                content = f.read()

                list_of_files.append(content)

    random.Random(seed).shuffle(list_of_files)

    test_index = int(len(list_of_files) * 0.8)
    train_split = " ".join(list_of_files[:test_index])
    val_split = " ".join(list_of_files[test_index:])

    return train_split, val_split


train_split, val_split = train_val_split(42)
full_text = train_split + val_split

vocabulary = sorted(set(full_text))

print("Vocabulary length: " + str(len(vocabulary)))

map_to_integer_id = {}
map_to_character = {}

for i in range(len(vocabulary)):
    map_to_integer_id[vocabulary[i]] = i
    map_to_character[i] = vocabulary[i]

def encode(string):
    list_of_integer_ids = []

    for char in string:
        list_of_integer_ids.append(map_to_integer_id[char])

    return torch.tensor(list_of_integer_ids, dtype=torch.long)

def decode(list_of_integer_ids):
    string = ""

    for id in list_of_integer_ids:
        string += map_to_character[int(id)]

    return string

print(encode("war"))
print(decode(encode("war")))


In [ ]:
def get_batch(data, batch_size, block_size):
    random_positions = [random.randint(0, len(data) - 1 - block_size) for _ in range(batch_size)]

    input_slices = []
    target_slices = []

    for position in random_positions:
        input_slices.append(data[position:position+block_size])
        target_slices.append(data[position+1:position+block_size+1])

    return torch.stack(input_slices), torch.stack(target_slices)


In [ ]:
data = torch.as_tensor(encode(train_split), dtype=torch.long)
val_data = torch.as_tensor(encode(val_split), dtype=torch.long)

inputs, targets = get_batch(data, batch_size=4, block_size=8)

print(inputs.shape)   # torch.Size([4, 8])
print(targets.shape)  # torch.Size([4, 8])
print(inputs)
print(targets)
print(decode(inputs[0].tolist())) 

In [ ]:
def generate(model, prompt_text, max_new_tokens): 
    encoded = encode(prompt_text)
    new_tokens = 0

    while new_tokens != max_new_tokens:
        prediciton = model(encoded[-block_size:].unsqueeze(0))

        logit_at_last_position = prediciton[0][-1]
        
        probs = torch.softmax(logit_at_last_position, dim=-1)

        probs_mask = torch.ones(probs.shape[-1]).bool()
        top10_indices = torch.topk(probs, 10).indices
        probs_mask[top10_indices] = False
        probs = probs.masked_fill(probs_mask, 0)

        next_char_int_id = torch.multinomial(probs, num_samples=1)
        
        new_tokens += 1
        
        encoded = torch.cat((encoded, next_char_int_id), dim=0)
    
    return decode(encoded)

In [ ]:
mlflow.set_experiment(experiment_name="transformer_char_based")

with mlflow.start_run(run_name="training"):
        batch_size=64
        block_size=128
        d_model=256
        num_heads=4
        num_blocks=6
        lr=3e-4
        num_steps=5000

        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("block_size", block_size)
        mlflow.log_param("d_model", d_model)
        mlflow.log_param("num_heads", num_heads)
        mlflow.log_param("num_blocks", num_blocks)
        mlflow.log_param("lr", lr)
        mlflow.log_param("num_steps", num_steps)

        loss_function = nn.CrossEntropyLoss()
        transformer = Transformer(vocab_size=len(vocabulary), d_model=d_model, num_heads=num_heads, num_blocks=num_blocks, seq_len=block_size)
        optimizer = torch.optim.AdamW(params=transformer.parameters(), lr=lr)

        for i in range(num_steps):
                inputs, targets = get_batch(data, batch_size=batch_size, block_size=block_size)

                forward_pass = transformer(inputs)

                targets_loss = targets.reshape(-1)
                forward_pass_loss = forward_pass.reshape(-1, forward_pass.size(-1))
                
                loss = loss_function(forward_pass_loss, targets_loss)

                mlflow.log_metric("train_loss", loss.item(), step=i)

                loss.backward()

                optimizer.step()

                optimizer.zero_grad()

                if i % 250 == 0:
                        transformer.eval()
                        losses = []

                        for _ in range(10):
                                inputs, targets = get_batch(val_data, batch_size=batch_size, block_size=block_size)

                                with torch.no_grad():
                                        forward_pass = transformer(inputs)

                                        targets_loss = targets.reshape(-1)
                                        forward_pass_loss = forward_pass.reshape(-1, forward_pass.size(-1))
                                        
                                        loss = loss_function(forward_pass_loss, targets_loss)

                                        losses.append(loss.item())
                        
                        final_loss = sum(losses) / len(losses)
                        mlflow.log_metric("val_loss", final_loss, step=i)
                        
                        transformer.train()
        
        mlflow.pytorch.log_model(pytorch_model=transformer, name="model")
        mlflow.log_text(generate(transformer, "War is", 500), "sample.txt")